# Quick start

Run all cells with the project dependencies installed. These are the two runnable examples from the [README](../README.md).

In [1]:
import sys
from pathlib import Path

# Works from the repository root or this notebook's directory.
root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "resource_estimation" / "clifford_verification").is_dir()
)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

## Clifford circuits

A CNOT and its H–CZ–H decomposition have the same channel.

In [2]:
import cirq
from resource_estimation.clifford_verification import cliffords_equal

a, b = cirq.LineQubit.range(2)
original = cirq.Circuit(cirq.CNOT(a, b))
optimized = cirq.Circuit(cirq.H(b), cirq.CZ(a, b), cirq.H(b))
assert cliffords_equal(original, optimized)
print("Equal:", cliffords_equal(original, optimized))

Equal: True


## T-state preparations

Declare a fresh resource wire with the compiler’s `ResourceStateTag`. The verifier matches T/T† declarations and compares the remaining channels with those wires treated as open inputs. The marker is an identity to ordinary Cirq simulation; it does not physically prepare a state.

In [3]:
import cirq
from resource_estimation.clifford_verification import cliffords_equal
from resource_estimation.compile_gateset.ppm_compiler import ResourceStateTag

data, resource = cirq.LineQubit.range(2)
# Declare a T-state preparation; do not apply a T gate to the circuit.
marker = cirq.I(resource).with_tags(ResourceStateTag(cirq.T, 0, 0, 1))
original = cirq.Circuit(marker, cirq.CNOT(data, resource))
optimized = cirq.Circuit(
    marker, cirq.H(resource), cirq.CZ(data, resource), cirq.H(resource)
)
assert cliffords_equal(original, optimized)
print("Equal with tagged resource:", cliffords_equal(original, optimized))

Equal with tagged resource: True


Use `cirq.T**-1` in the tag for T†. The marker must be the wire’s first operation. An actual T gate in the circuit is rejected; a failed resource certification raises an error rather than claiming inequality.